# 🐱 Meow Decoder: try it

Record or upload a meow and see the three likeliest reasons behind it, with honest percentages.

It doesn't translate; cats don't meow in sentences. It guesses which situation a meow sounds like (waiting for food, enjoying attention, or stressed somewhere new). On cats it has never heard, its first guess is right about 2 times in 3, and the right answer is in its top two about 9 times in 10.

**Run the cells top to bottom** (Runtime → Run all works too). Installing takes a minute or two. Code and benchmark: [github.com/ATHARV-IIT-BOMBAY/meow-decoder](https://github.com/ATHARV-IIT-BOMBAY/meow-decoder)

## 1. Install

In [ ]:
%pip install -q "meow-decoder @ git+https://github.com/ATHARV-IIT-BOMBAY/meow-decoder"

## 2. Load the decoder

Downloads the audio models once (about a minute).

In [ ]:
import io, base64
from pathlib import Path
import numpy as np
from IPython.display import Audio, display
import meow_decoder
from meow_decoder import Decoder

decoder = Decoder()  # Decoder("ast") for the commercial-safe encoder
decoder.embed(np.zeros(16000, np.float32))  # load models now

def show(result, name=""):
    print(name + ("" if result["is_cat"] else "  (doesn't sound much like a cat: try closer, away from fans and traffic)"))
    for r in result["results"]:
        print(f"  {r['p']:>4.0%}  {r['emoji']} {r['title']}: \"{r['line']}\"")
    heard = [n for k, n in [("Purr", "purring"), ("Hiss", "hissing"), ("Caterwaul", "yowling")] if result["sounds"][k] >= 0.2]
    if heard:
        print("  also heard:", ", ".join(heard))

## 3. Try three meows from the research dataset

In [ ]:
samples = Path(meow_decoder.__file__).parent / "static" / "samples"
for name in ["hungry", "brushed", "alone"]:
    path = samples / f"{name}.wav"
    display(Audio(str(path)))
    show(decoder.predict(path), f"{name}.wav")

## 4. Upload your own recording

Any audio or video file: a voice memo, a phone video of your cat, a .wav or .mp3.

In [ ]:
try:
    from google.colab import files
    for name, data in files.upload().items():
        display(Audio(data=data, autoplay=False))
        show(decoder.predict(data), name)
except ImportError:
    print("Uploading needs Google Colab. Locally, use: decoder.predict('your-file.m4a')")

## 5. Record a meow right here

Run the cell, allow the microphone, hold your device near the cat. It records for 6 seconds.

In [ ]:
RECORD_JS = '''
async function record(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream), chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const done = new Promise(r => rec.onstop = r);
  rec.start(); await new Promise(r => setTimeout(r, ms)); rec.stop(); await done;
  stream.getTracks().forEach(t => t.stop());
  const buf = await new Blob(chunks).arrayBuffer();
  let s = ''; new Uint8Array(buf).forEach(b => s += String.fromCharCode(b));
  return btoa(s);
}
'''
try:
    from google.colab.output import eval_js
    print("Recording for 6 seconds…")
    data = base64.b64decode(eval_js(RECORD_JS + "record(6000)"))
    show(decoder.predict(data), "your recording")
except ImportError:
    print("Recording needs Google Colab.")

## 6. Open the full app (optional)

The same app as `meow serve`, with the reacting cat and *teach it your cat*. It opens in a new tab and stops when this notebook does, taking any cats you added with it.

In [ ]:
import threading, uvicorn
from meow_decoder.server import build_app, default_home

server = uvicorn.Server(uvicorn.Config(build_app(decoder, default_home(), contrib=False), host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8000, anchor_text="Open Meow Decoder")
except ImportError:
    print("Running at http://127.0.0.1:8000")

---
Its percentages are honest, but it only knows three situations and can't detect pain or illness. If a cat seems unwell, see a vet.

Data: [CatMeows](https://zenodo.org/records/4008297) (CC BY 4.0). Default encoder: Earth Species Project BEATs (CC BY-NC-SA 4.0, non-commercial).